# CNN · ранний базовый эксперимент

Ранний вариант с собственным CSV-загрузчиком и нелогарифмированной целевой массой. Сохранён отдельно от основных запусков: протокол и метки отличаются от `notebooks/training/`.

Пути определяются автоматически от корня репозитория. См. [воспроизведение](../../docs/reproduction.md) и [описание результатов](../../docs/results.md).


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
    and (path / "src" / "galaxy_mass_estimation").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Импорты


In [ ]:
import numpy as np
import sys
import os
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
import pandas as pd

output_dir = str(PROJECT_ROOT / "results/exploration/cnn_baseline") + "/"
os.makedirs(output_dir, exist_ok=True)

## Воспроизводимость


In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    os.environ["PYTHONHASHSEED"] = str(seed)

## Окружение и параметры


In [ ]:
set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EPOCHS = 100
LR = 1e-3
BATCH_SIZE = 128

## Архитектура модели


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
        )

    def forward(self, images, redshift):
        return self.cnn(images)

## Модель и оптимизатор


In [ ]:
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
loss_fn = torch.nn.MSELoss()

## Загрузка данных


In [ ]:
class ClusterDataset(Dataset):
    def __init__(self, images_dir, indices, csv_path):
        self.images_dir = images_dir
        self.indices = indices
        self.df = pd.read_csv(csv_path)

        self.valid_samples = []
        for idx in indices:
            file_name = f"{idx+1}.npy"
            file_path = os.path.join(images_dir, file_name)

            if os.path.exists(file_path):
                self.valid_samples.append((file_path, idx))
            else:
                print(f"Пропускаем отсутствующий файл: {file_path}")
                continue

        print(f"ГОТОВО: {len(self.valid_samples)} файлов из {len(indices)}")

    def __len__(self):
        return len(self.valid_samples)

    def __getitem__(self, idx):
        file_path, csv_idx = self.valid_samples[idx]
        image = np.load(file_path).astype(np.float32)
        row = self.df.iloc[csv_idx]
        redshift = np.array([row["redshift"]], dtype=np.float32)
        target = np.float32(row["M500cUncorr"])
        return torch.tensor(image), torch.tensor(redshift), torch.tensor(target)

## Подготовка выборок


In [ ]:
train_indices = np.load(PROJECT_ROOT / "data/splits/raw/train_indices.npy")
test_indices = np.load(PROJECT_ROOT / "data/splits/raw/test_indices.npy")

train_dataset = ClusterDataset(
    images_dir=str(PROJECT_ROOT / "data/processed/images"),
    indices=train_indices,
    csv_path=str(PROJECT_ROOT / "data/raw/clusters_selected.csv"),
)

test_dataset = ClusterDataset(
    images_dir=str(PROJECT_ROOT / "data/processed/images"),
    indices=test_indices,
    csv_path=str(PROJECT_ROOT / "data/raw/clusters_selected.csv"),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

## Обучение


In [ ]:
def train_model(model, train_loader, test_loader, epochs, device):
    model.train()
    losses = []

    for epoch in range(epochs):
        print(epoch)
        start_t = time.time()
        total_loss = 0
        batch_count = 0

        for img, scalars, target in train_loader:
            img = img.to(device)
            target = target.to(device).unsqueeze(1)

            optimizer.zero_grad()
            pred = model(img, scalars)
            loss = loss_fn(pred, target)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            batch_count += 1

        avg_loss = total_loss / batch_count
        losses.append(avg_loss)
        scheduler.step()

        if epoch == 0:
            print(f"Epoch 1 time: {time.time() - start_t:.2f} сек")

        if (epoch + 1) % 10 == 0:
            eval_metrics = evaluate_model(model, test_loader, device)

            print(f"\nEPOCH {epoch+1}/{epochs}")
            print(f"Train Loss: {avg_loss:.4f}")
            print(f"Val MAE: {eval_metrics['mae']:.4f}")
            print(f"Val Relative Error: {eval_metrics['rel_error']:.4f}")
            print(f"Max Relative Error: {eval_metrics['max_rel_error']:.4f}")
            print("-" * 50)

    return losses


def evaluate_model(model, test_loader, device):
    model.eval()
    preds, targets = [], []

    with torch.no_grad():
        for img, scalars, target in test_loader:
            img = img.to(device)
            target = target.to(device).unsqueeze(1)

            pred = model(img, scalars)
            preds.append(pred)
            targets.append(target)

    model.train()

    preds = torch.cat(preds)
    targets = torch.cat(targets)

    mae = torch.abs(targets - preds)
    relative_errors = mae / torch.abs(targets)

    return {
        "mae": mae.mean().item(),
        "max_mae": mae.max().item(),
        "rel_error": relative_errors.mean().item(),
        "max_rel_error": relative_errors.max().item(),
        "preds": preds,
        "targets": targets,
    }


print("ура обучение...")
losses = train_model(model, train_loader, test_loader, EPOCHS, device)

print("\n" + "=" * 60)
print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ:")
final_metrics = evaluate_model(model, test_loader, device)
print(f"Средняя MAE: {final_metrics['mae']:.4f}")
print(f"Максимальная MAE: {final_metrics['max_mae']:.4f}")
print(f"Средняя относительная ошибка: {final_metrics['rel_error']:.4f}")
print(f"Максимальная относительная ошибка: {final_metrics['max_rel_error']:.4f}")

## Оценка и графики


In [ ]:
losses = np.asarray(losses)

model.eval()
preds = []
targets = []

with torch.no_grad():
    for img, scalars, target in test_loader:
        img = img.to(device)
        scalars = scalars.to(device)
        target = target.to(device).unsqueeze(1)

        pred = model(img, scalars)

        preds.append(pred.cpu())
        targets.append(target.cpu())

preds = torch.cat(preds)
targets = torch.cat(targets)

mae = torch.abs(targets - preds)
mean_mae = mae.mean().item()
max_mae = mae.max().item()

relative_errors = mae / torch.abs(targets)
mean_rel_error = relative_errors.mean().item()
max_rel_error = relative_errors.max().item()

print(f"EPOCH={epoch+1}")
print(f"Средняя MAE: {mean_mae:.4f}")
print(f"Максимальная MAE: {max_mae:.4f}")
print(f"Средняя относительная ошибка: {mean_rel_error:.4f}")
print(f"Максимальная относительная ошибка: {max_rel_error:.4f}")

np.save(output_dir + "losses.npy", losses)
np.save(output_dir + "targets.npy", targets)
np.save(output_dir + "predictions.npy", preds)
torch.save(model.state_dict(), output_dir + "model.pth")